# ⚡ Vigyan-2B: Ultra-Fast Sovereign Edge STEM Model (Non-Commercial Demo)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyansh001boy-tech/vigyan-7b-demo/blob/main/vigyan_2b_demo.ipynb)
&nbsp;[![License: CC BY-NC 4.0](https://img.shields.io/badge/License-CC_BY--NC_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
&nbsp;[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Vigyan--2B--STEM--Instruct-yellow)](https://huggingface.co/shreyansh12183/Vigyan-2B-STEM-Instruct-v1)

---

### 🌟 Overview
**Vigyan-2B-STEM-Instruct-v1** is India's sovereign edge Small Language Model (SLM), fine-tuned for high-speed, on-device STEM reasoning across Circuit Physics, Logic Design, and Applied Mathematics.

> **Hardware Requirement:** Free Google Colab **Tesla T4 GPU** (Only ~1.8 GB VRAM in 4-bit NF4 precision) or high-RAM CPU.
> **Inference Speed:** ~50+ tokens/second for instant sub-second answers.
> **License:** Creative Commons Attribution-NonCommercial 4.0 International (**CC BY-NC 4.0**). Free for academic and research use.

### Step 1: Install Dependencies (Runs in ~25 seconds)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes gradio
print('✅ Environment ready.')

### Step 2: Load Vigyan-2B in 4-bit Precision (~1.8 GB VRAM)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = 'shreyansh12183/Shreyansh-STEM-AI-2B-v3'
ADAPTER_ID = 'shreyansh12183/Vigyan-2B-STEM-Instruct-v1'

print(f'⚡ PyTorch version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'⚡ Device: {torch.cuda.get_device_name(0)}')

print('📥 Loading Tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print('📥 Loading 2B Base Weights in 4-bit NF4...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16
)

print('📥 Attaching Vigyan-2B LoRA Adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model.eval()

if hasattr(model, 'generation_config') and hasattr(model.generation_config, 'stop_strings'):
    model.generation_config.stop_strings = None

print('🎉 Vigyan-2B Model Loaded Successfully!')

### Step 3: Launch Interactive Gradio Web UI (with Public Share Link)

In [ ]:
import gradio as gr
import time

def solve_stem_problem(question, max_tokens, temperature):
    if not question.strip():
        return 'Please enter a valid STEM problem.'
    
    prompt = f'<|im_start|>user\n{question.strip()}\nSolve step by step. State your final answer clearly in \\boxed{{number}}.<|im_end|>\n<|im_start|>assistant\n'
    
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda' if torch.cuda.is_available() else 'cpu')
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            tokenizer=tokenizer,
            max_new_tokens=int(max_tokens),
            do_sample=True if temperature > 0.05 else False,
            temperature=float(temperature) if temperature > 0.05 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    dt = time.time() - t0
    
    gen_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    
    tokens_count = len(gen_tokens)
    speed = round(tokens_count / dt, 1) if dt > 0 else 0.0
    
    footer = f'\n\n---\n⚡ **Latency:** {dt:.2f}s | **Tokens:** {tokens_count} | **Throughput:** {speed} tok/s'
    return response + footer

examples = [
    ['Two resistors R1 = 150 ohms and R2 = 300 ohms are connected in parallel. Calculate equivalent resistance in ohms.'],
    ['A circuit has Thevenin voltage V_th = 12 V and Thevenin resistance R_th = 3 ohms. Calculate maximum power transfer to a load in Watts.'],
    ['A mass m = 5 kg slides down a frictionless 30-degree incline. What is its acceleration in m/s^2? Use g = 9.8.'],
    ['A CMOS logic gate has tpLH = 45 ps and tpHL = 35 ps. Calculate average propagation delay in picoseconds.'],
    ['Find the determinant of the 2x2 matrix [[4, 5], [2, 3]]. State the exact number in \\boxed{}.']
]

with gr.Blocks(theme=gr.themes.Soft(primary_hue='emerald')) as demo:
    gr.Markdown(
        '''
        # ⚡ Vigyan-2B Edge STEM AI: Live Interactive Reasoning
        ### Ultra-Fast Edge Language Model for Embedded & Applied Science
        *Vigyan AI — Varanasi, India | Non-Commercial Research Release (CC BY-NC 4.0)*
        '''
    )
    
    with gr.Row():
        with gr.Column(scale=4):
            input_box = gr.Textbox(
                label='Enter STEM Problem',
                placeholder='Type your physics, circuit, or algebra question...',
                lines=4
            )
            with gr.Row():
                max_toks = gr.Slider(minimum=64, maximum=512, value=256, step=32, label='Max Tokens')
                temp = gr.Slider(minimum=0.0, maximum=1.0, value=0.1, step=0.05, label='Temperature')
            submit_btn = gr.Button('⚡ Compute Solution with Vigyan-2B', variant='primary')
            
            gr.Examples(
                examples=examples,
                inputs=input_box,
                label='Quick One-Click Test Cases'
            )
            
        with gr.Column(scale=5):
            output_box = gr.Markdown(label='Model Output')
            
    submit_btn.click(
        fn=solve_stem_problem,
        inputs=[input_box, max_toks, temp],
        outputs=output_box
    )

print('🚀 Launching Gradio Web UI with public share link...')
demo.launch(share=True, debug=False)
